# Notebook 2 — MLflow Model Registry

This notebook demonstrates the **MLflow Model Registry** for governing ML model lifecycle.

**What you'll learn:**
- How models get registered automatically after training
- Viewing model versions and their metadata
- Assigning aliases (`champion`, `challenger`) to model versions
- Loading a model from the registry by alias
- Running evaluation and conditionally promoting to champion
- Serving a model with `mlflow models serve`

**Prerequisites:** At least one training run completed (run Notebook 1 first)

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import mlflow
from mlflow import MlflowClient
from dotenv import load_dotenv

load_dotenv()
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000'))

MODEL_NAME = 'mnist-cnn-model'
client = MlflowClient()

print('MLflow tracking URI:', mlflow.get_tracking_uri())

## 1. List All Registered Model Versions

In [ ]:
import pandas as pd

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
print(f'Total versions of "{MODEL_NAME}": {len(versions)}')

if versions:
    rows = []
    for v in versions:
        rows.append({
            'version': v.version,
            'run_id': v.run_id[:12] + '...',
            'aliases': v.aliases,
            'tags': dict(v.tags),
            'created_at': pd.Timestamp(v.creation_timestamp, unit='ms'),
        })
    display(pd.DataFrame(rows))
else:
    print('No versions found. Run training first (Notebook 1).')

## 2. Evaluate and Promote to Champion

In [ ]:
from training.config import TrainingConfig
from training.evaluate import promote_to_champion

if versions:
    # Use the latest version's run_id for promotion
    latest_version = versions[-1]
    print(f'Evaluating version {latest_version.version} (run_id={latest_version.run_id[:12]}...)')
    
    cfg = TrainingConfig()
    promoted = promote_to_champion(
        model_name=MODEL_NAME,
        run_id=latest_version.run_id,
        min_accuracy=0.95,  # lower threshold for quick demo
        cfg=cfg,
    )
    print(f'Promoted to champion: {promoted}')
else:
    print('No model versions available.')

## 3. Load the Champion Model for Inference

In [ ]:
import torch
from torchvision import datasets, transforms

# Load champion model from registry
model_uri = f'models:/{MODEL_NAME}@champion'
model = mlflow.pytorch.load_model(model_uri)
model.eval()
print(f'Loaded champion model from: {model_uri}')

# Run inference on a few test images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

images, labels = zip(*[test_dataset[i] for i in range(8)])
batch = torch.stack(images)

with torch.no_grad():
    logits = model(batch)
    predictions = logits.argmax(dim=1)

for i, (pred, label) in enumerate(zip(predictions.tolist(), labels)):
    status = '✓' if pred == label else '✗'
    print(f'  Sample {i}: predicted={pred}  actual={label}  {status}')

## 4. Add a Challenger (Tag a Version for A/B Comparison)

In [ ]:
# If there are 2+ versions, tag the second-best as 'challenger'
versions = client.search_model_versions(f"name='{MODEL_NAME}'")

if len(versions) >= 2:
    # Find version without champion alias
    challengers = [v for v in versions if 'champion' not in v.aliases]
    if challengers:
        challenger = challengers[0]
        client.set_registered_model_alias(
            name=MODEL_NAME,
            alias='challenger',
            version=challenger.version,
        )
        print(f'Version {challenger.version} tagged as challenger')
        print(f'A/B compare: champion vs challenger at http://localhost:5000')
    else:
        print('All versions are champions — train another run to create a challenger.')
else:
    print('Only one version exists. Run training again with different hyperparameters to create a challenger.')

## 5. Serve the Champion Model

Run this command in your terminal to start an inference server:

```bash
cd model-training
uv run mlflow models serve \
  -m 'models:/mnist-cnn-model@champion' \
  -p 8080 \
  --env-manager local
```

Then send a prediction request:

In [ ]:
# Example of what the serving request looks like (requires server running on port 8080)
import json
import numpy as np

# Build a sample input in MLflow serving format
sample_input = batch[:2].numpy()  # shape (2, 1, 28, 28)
payload = {'inputs': sample_input.tolist()}

print('Sample serving payload structure:')
print(f'  inputs shape: {sample_input.shape}')
print(f'  JSON keys:    {list(payload.keys())}')
print()
print('To call the live server:')
print("  curl -X POST http://localhost:8080/invocations \\")
print("    -H 'Content-Type: application/json' \\")
print("    -d '{\"inputs\": [[...]]}'")

## Summary

The MLflow Model Registry provides:

| Feature | Benefit |
|---|---|
| Version tracking | Full history of every trained model |
| Aliases | Named pointers (`champion`, `challenger`) for safe rollouts |
| Lineage | Every version links back to its training run + data |
| Serving | One command to deploy any registered version |

This is the difference between **"we have a model"** and **"we have a production ML system"**.